<h2 align="center"><b>Analysis of MDS Trajectory: AmberTools Tutorials </b></h2>

<div align="center">
  <img src="https://github.com/SHEDOOMTC/Basic_Moco_2/raw/main/Assets/Moco_2_visuals.png" alt="Description" width="400"/>
</div>

### **About and Usage**

In this tutorial, we will walk through processing, manipulation and aanalysis of **trajectory** from molecular dynamics simulation run in **[AmberTools](https://ambermd.org/AmberTools.php)**. This is a follow-up from the practical session **[Practicals_AmberTools.ipynb](Link)** 

This is a practical session follow up of the course on **"[Basic Molecular Modelling Course for Chemists and Life Sciences II](https://github.com/SHEDOOMTC/Basic_Moco_2)** ***(Basic Moco II)***"

Cite this tutorial as **"""""**

### **1. CPPTRAJ**

Cpptraj is the central tool for manipulating and processing trajectories in AmberTools. It is a successor fo the popular **[PTRAJ from Roe & Cheatham](https://doi.org/10.1021/ct400341p)**

It is bundled into AmberTools on istallation but can also be installed separately for use. It allows both a command line interface and a python wrapper **[Pytraj](https://github.com/Amber-MD/pytraj)**

CPPTRAJ supports many popular MD software packages including Amber, **[CHARMM](https://www.charmm.org/charmm/)**, **[Gromacs](http://www.gromacs.org/)**, and **[NAMD](http://www.ks.uiuc.edu/Research/namd/)**.

For the analysis here, we will use the trajectories from the production run of the tutorial on AmberTools "**[Practicals_AmberTools.ipynb](Link)**. This trajectory contains 10000 frames of ligand-bound **[3HTB](https://www.rcsb.org/structure/3HTB)** (T4 lysozyme).

In [ ]:
#Import Libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import nglview as nv
import mdtraj as md

In [ ]:
#To understand its functionalities, run the following command in your terminal:
!cpptraj -h

In [ ]:
## The popular welcome message that gets printed when you run cpptraj -h is as follows:

#       CPPTRAJ: Trajectory Analysis. V6.29.13 (AmberTools)
#           ___  ___  ___  ___
#            | \/ | \/ | \/ |
#           _|_/\_|_/\_|_/\_|_
#
#           | Date/time: 10/10/26 23:07:50
#           | Available memory: 14.216 GB
#           >

While running interactively is great, it is better run by placing all the command into a a script and executing

In [ ]:
%%bash
#cat > input_commands.in << EOF
#parm system.prmtop
#trajin md1.nc
#trajin md2.nc
#trajin md3.nc
#trajout combined.nc netcdf
#run
#EOF

In [ ]:
# Then run it by executing the following command in your terminal:
# !cpptraj -i input_commands.in

### **2. Preprocessing Trajectories**

- **Combining Trajectories**

Trajectories are like movies, which are ensembles of snapshots or frames. It can therefore be stuck together, cut, split etc. Many MD runs are done in batches and thus needs a trajectory combination at the end of the run to achieve the required simulation time. 

**Note**: The temporal sequence of the trajectories must be maintained

*This uses the same set of trajectories from the mdrun in* **[Practicals_AmberTools.ipynb]()**. *Due to size issue, download them from* **[Zenodo]()** *and copy them into your working directory*

The files include trajectories named *traj_1.nc traj_2.nc traj_3.nc traj_4.nc*, and the associated topology, *complex.top*. These trajectories each hold 2500 frames; totalling 10,000 frames.

**The trajectories from Zenodo have been stripped of water and ions aprior, so the strip commands below are for learning sake**

In [ ]:
# If you ran the mdrun successfuuly, you can also copy from your AmberTools_run folder
!cp ../AmberTools_run/mdrun/traj*.nc && cp ../AmberTools_run/mdrun/complex.top .


In [ ]:
#If you want to skip the whole analysis step, then copy the full analysis from ../AmberTools_Analysis_done 
#!cp ../AmberTools_Analysis_done/* .

#Then make only the necessary plots and figures for your report.

<font color="darkred">*What is a multimodel PDB? Can you create a 10 frame trajectory file by concatenating ensembles of same protein in a PDB file? What would separate one frame from another?*</font>

- **Lets combine the trajectories**

After reading in the trajectories, we fix molecules that moved across periodic boundaries using the *autoimage*. Then you remove artifacts using *unwrap*, before centering the frames at the centre of the protein and then reimage the solvents around it.

In [ ]:
%%bash
cat > combine_traj.in << EOF
parm complex.top
trajin traj_1.nc
trajin traj_2.nc
trajin traj_3.nc
trajin traj_4.nc
autoimage
unwrap
center :1-163
image origin center
trajout traj_tot.nc netcdf #save output
run
EOF

In [ ]:
#Run it in your terminal:
!cpptraj -i combine_traj.in

<font color="darkred">*What would happen if you miss the temporal order of the trajectories you want to combine? How will it impact the interpretability of your results?*</font>

Many times, it is usuall to remove all or some water molecules and ions during the trajectory combination. This makes the combined trajectory lighter for downstream tasks. Water already accounts for about 40-60% of the weight of the system. Especially in cases when there is no focus on water hydration or specific ion movements. Here the **strip command** comes into play. We will produce another combined, but stripped trajectory called **traj_strip.nc** using another input file combine_traj_strip.in.

In [ ]:
%%bash
cat > combine_traj_strip.in << EOF
parm complex.top
trajin traj_1.nc
trajin traj_2.nc
trajin traj_3.nc
trajin traj_4.nc
autoimage
unwrap
center :1-163
image origin center
strip :WAT,Na+,Cl- outprefix stripped
trajout traj_strip.nc netcdf #save output
run
EOF

# The outprefix stripped will also create a topology files with the water molecules and ions stripped from the system. 
#The stripped topology file will be named stripped_complex.top.

In [ ]:
#Run it in your terminal:
!cpptraj -i combine_traj_strip.in

Trajectories can also be read at intervals, or at an offset from the start or till the end. We will read every 20th frame from the combined and stripped trajectory. This will amount to 500 frames

**From here, we will use the stripped topology (stripped_complex.top)**

In [ ]:
#If you are using the the trajectory data from Zenodo, then the topology is already stripped, so run this
#!cp complex.top stripped.complex.top

In [ ]:
%%bash
cat > traj_20_steps.in << EOF
parm stripped.complex.top
trajin traj_strip.nc 1 last 20
trajout traj_strip_20steps.nc netcdf #save output
run
EOF

In [ ]:

#Then Run it in your terminal:
!cpptraj -i traj_20_steps.in

<font color="darkred">*How would you collect only the last 2ns of your entire simulation using Cpptraj command?*</font>

- **Ligand Trajectory**

And finally, let us generate a ligand trajectory from the full trajectory using same strip command and an input file. *Our ligand is the resnumber 164*. So we stripp everything but resnumber 164 (!164). 

In [ ]:
%%bash
cat > ligand_traj.in << EOF
parm stripped.complex.top
trajin traj_strip.nc
strip :164
trajout ligand_traj.nc netcdf #save output
run
EOF

In [ ]:
#Then Run it in your terminal:
!cpptraj -i ligand_traj.in

### **3. RMSD/Alignment**

**[RMSD (Root Mean Square Deviation)](https://doi.org/10.1021/acs.jctc.7b00028)** measures the average distance between the atoms of *superimposed or aligned structures*, typically comparing a trajectory frame to a *reference structure* (like a crystal structure or the first frame of the trajectory). It is a fundamental metric in molecular dynamics for assessing system *stability and convergence*. A plot of RMSD over time reveals whether the protein has equilibrated. A rapid initial rise usually indicates structural relaxation, while a stable plateau suggests the system has reached a steady state. Sudden spikes or persistent drift imply conformational changes, instability, or that the simulation has not yet converged.

We will compute the RMSD over the backbone proteins after allignment, using the first frame as the reference. Usually the alpha carbon (Cα) is good enough; including other heavy atoms might increase precision but also demand more compute. The output is usually saved in the *DAT* format.

In [ ]:
%%bash
cat > RMSD.in << EOF
parm stripped.complex.top
trajin traj_strip.nc
rms first out rmsd.dat @CA
run
EOF

#Then Run it in your terminal:
cpptraj -i RMSD.in

- **RMSD Plot**

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

data = np.loadtxt('rmsd.dat', comments='#')
time = data[:, 0] / 1000 #Divide frames by 1000 to convert to nanoseconds (ns) 
rmsd = data[:, 1]

plt.figure(figsize=(8, 5))
plt.plot(time, rmsd, color='blue', linewidth=1.5)
plt.xlabel('Time (ns)', fontsize=12)
plt.ylabel('RMSD (Å)', fontsize=12)
plt.title('Protein Backbone RMSD', fontsize=14)
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('rmsd_plot.png', dpi=300)
plt.show()

<font color="darkred">*What is the optimum RMSD? Can RMSD alone be used to judge the quality of your simulation?*</font>

- **Ligand RMSD**

Ligand RMSD measures the structural deviation of a small molecule (drug/ligand) relative to a reference structure over time. Unlike protein RMSD, which assesses global folding stability, ligand RMSD is used to evaluate binding stability and pose retention within the active site. In CPPTRAJ, you typically calculate this by selecting only the ligand atoms (e.g., :LIG) and often aligning the trajectory on the protein backbone first. A low, stable ligand RMSD indicates the ligand remains tightly bound in its original pose, while a rising RMSD suggests it is drifting or undergoing conformational changes.

In [ ]:
%%bash
cat > RMSD_ligand.in << EOF
parm stripped.complex.top
trajin traj_strip.nc
rms first @CA
rms ligand first :164 out ligand_rmsd.dat
run
EOF

#Then Run it in your terminal:
cpptraj -i RMSD_ligand.in

- **Ligand RMSD Plot**

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

data = np.loadtxt('ligand_rmsd.dat', comments='#')
time = data[:, 0] / 1000 #Divide frames by 1000 to convert to nanoseconds (ns) 
rmsd = data[:, 1]

plt.figure(figsize=(8, 5))
plt.plot(time, rmsd, color='blue', linewidth=1.5)
plt.xlabel('Time (ns)', fontsize=12)
plt.ylabel('RMSD (Å)', fontsize=12)
plt.title('Ligand RMSD', fontsize=14)
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('ligand_rmsd_plot.png', dpi=300)
plt.show()

<font color="darkred">*Is ligand RMSD correlated to protein backbone RMSD? What do you think about the ligand RMSD from the plots above?*</font>

### **4. RMSF**

**[RMSF (Root Mean Square Fluctuation)]()** measures the average deviation of an atom (or residue) from its mean position over the course of a trajectory. Unlike RMSD, which shows global structural changes over time, RMSF identifies local flexibility and dynamics. High RMSF values indicate highly flexible regions (like loops or terminal tails), while low values indicate rigid, stable regions (like secondary structures or the binding pocket). It is an essential metric for understanding protein stability and identifying which specific residues undergo the most movement during the simulation.

Can be averaged for each residues (using the “byres” keyword); the defaults computes for each atom. The crystallographic equivilant of bfactors can also be computed. We will compute the RMSF over all the residues of our protein.

In [ ]:
%%bash
cat > RMSF.in << EOF
parm stripped.complex.top
trajin traj_strip.nc
reference traj_strip.nc 1 1
rms reference @CA
atomicfluct out bfactor.dat :1-163@CA byres bfactor
atomicfluct out rmsf.dat :1-163@CA byres
run
EOF

#Then Run it in your terminal:
cpptraj -i RMSF.in

- **RMSF Plot**

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

data = np.loadtxt('rmsf.dat', comments='#')
residue = data[:, 0]
rmsf = data[:, 1]
plt.figure(figsize=(8, 5))
plt.plot(residue, rmsf, color='crimson', linewidth=1.5)
plt.title('Root Mean Square Fluctuation (RMSF)', fontsize=14)
plt.xlabel('Residue Number', fontsize=12)
plt.ylabel('RMSF (Å)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('rmsf_plot.png', dpi=300)
plt.show()

<font color="darkred">*What regions of our proteins are least stable throughout the simulation?*</font>

<font color="darkred">*Why is there no time-component in the RMSF plot? How would you introduce a time component? Heard of trajectory maps?*</font>

### **5. ROG and SASA**

**[Radius of Gyration (Rog)]()** measures the compactness of a molecule. It is the root-mean-square distance of the atoms from their common center of mass. For proteins, Rog is used to monitor folding and unfolding. A stable, properly folded protein will have a relatively constant Rg. If the protein begins to unfold or lose its structural integrity, the Rog value will increase. It is an excellent indicator of overall structural stability during the simulation.

**[SASA (Solvent Accessible Surface Area)]()** calculates the surface area of a molecule that is accessible to a solvent (like water). It is typically computed by "rolling" a probe sphere (usually representing a water molecule, radius ~1.4 Å) over the protein's van der Waals surface. SASA is crucial for understanding protein folding, stability, and interactions. A drop in SASA often indicates the protein is folding or compacting, burying hydrophobic residues away from water. Conversely, an increase suggests unfolding or the exposure of buried regions.

In [ ]:

%%bash
cat > ROG.in << EOF
parm stripped.complex.top
trajin traj_strip.nc
reference traj_strip.nc 1 1
rms reference @CA
radgyr :1-163@CA out rog.dat
run
EOF

#Then Run it in your terminal:
cpptraj -i ROG.in

- **ROG Plot**

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

data = np.loadtxt('rog.dat', comments='#')

time_ns = data[:, 0] / 1000 
rg = data[:, 1]

plt.figure(figsize=(8, 5))
plt.plot(time_ns, rg, color='forestgreen', linewidth=1.5)
plt.title('Radius of Gyration (Rg)', fontsize=14)
plt.xlabel('Time (ns)', fontsize=12)
plt.ylabel('Rg (Å)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('rg_plot.png', dpi=300)
plt.show()

Since SASA is so compute intensive, we will run it over a few frames (skipped frames)

In [ ]:
%%bash
cat > SASA.in << EOF
parm stripped.complex.top
trajin traj_strip.nc 1 last 20
reference traj_strip.nc 1 1
rms reference @CA
surf :1-163 out sasa.dat
run
EOF

#Then Run it in your terminal:
cpptraj -i SASA.in

- **SASA Plot**

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

data = np.loadtxt('sasa.dat', comments='#')
time_ns = data[:, 0] / 1000 # Convert frames to ns
sasa = data[:, 1]
plt.figure(figsize=(8, 5))
plt.plot(time_ns, sasa, color='royalblue', linewidth=1.5)
plt.title('Solvent Accessible Surface Area (SASA)', fontsize=14)
plt.xlabel('Time (ns)', fontsize=12)
plt.ylabel('SASA (Å²)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('sasa_plot.png', dpi=300)
plt.show()

<font color="darkred">*Is ROG and SASA related? If yes, in what ways?*</font>

<font color="darkred">*Why are active sites usually buried deep inside the protein?*</font>

### **6. Distances, Angles, Dihedrals**

**[Distance]()** measures the straight line gap between the centers of two atoms or groups of atoms (Centre of Mass --COM). Distances are used to monitor specific interactions over time. For example, you might measure the distance between a protein residue and a ligand to see if a hydrogen bond is stable, or the distance between two domains to see if they are moving closer together or further apart.

The **[angle]()** (in degrees or radians) is formed by three connected atoms (Atom A – Atom B – Atom C), where Atom B is the vertex. Angles measure the bending or "flexing" of chemical bonds. Monitoring an angle can tell you if a specific part of a molecule is bending or distorting during the simulation.

**[Dihedrals]()** (Torsion Angles)is the angle formed by four connected atoms (Atom A – B – C – D). It measures the rotation around the B-C bond.They are critical for defining molecular conformation. The famous **Phi (φ)** and **Psi (ψ)** backbone dihedrals determine protein secondary structure, while side-chain dihedrals (like Chi1 and Chi2) dictate how a residue's side chain is positioned. Monitoring them shows if a molecule is flipping between different 3D shapes (conformations).

The active site of 3HTB is made of well structured helices. To aseess distances, angles and torsion, lets look at the less helical N-terminal part of the molecule. We will measure the distance between Alpha carbons of Glu22 and Lys35 at the tips of the active site facing loops; angle between Alpha carbons of Glu22, Lys35 and Gly30; and torsion between alpha carbons of Glu22, Lys35, Gly30 and Gly56.

*We will use a single script*.

In [ ]:
%%bash
cat > dist_ang_tors.in << EOF
parm stripped.complex.top
trajin traj_strip.nc
distance dist :22@CA :35@CA out distance.dat #DISTANCE
angle angle :22@CA :35@CA :30@CA out angle.dat #ANGLE
dihedral phi :22@C :35@N :30@CA :56@C out dihedral.dat #DIHEDRAL
run
EOF

#Then Run it in your terminal:
cpptraj -i dist_ang_tors.in

- **Plots**

For the plots below, edit the filename with the actual filenames for distance angles and torsion.

In [ ]:
def plot(filename, ylabel,
         title=None,
         color='purple',
         save=True):

    data = np.loadtxt(filename, comments='#')

    time_ns = data[:, 0] / 1000
    y = data[:, 1]

    plt.figure(figsize=(8, 5))
    plt.plot(time_ns, y, color=color, linewidth=1.5)

    plt.title(title or filename, fontsize=14)
    plt.xlabel('Time (ns)', fontsize=12)
    plt.ylabel(ylabel, fontsize=12)

    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()

    if save:
        plt.savefig(f'{filename}_plot.png', dpi=300)

    plt.show()

In [ ]:
#Plot the distance
plot("distance.dat", "Distance (Å)")

In [ ]:
#Plot the Angle
plot("angle.dat", "Angle (°)")

In [ ]:
#Plot the Dihedral
plot("dihedral.dat", "Dihedral Angle (°)")

To measure an angle or dihedral, **the order of the atoms matters**. If you mix up the order, the angle will be wrong (though for dihedrals, a reverse order will just give you a negative value of the same angle)

<font color="darkred">*What did you observe about the angles, torsion and distances during the simulation?*</font>

<font color="darkred">*Does the order of the atoms (masks) in the dist_ang_tors.in matter for the distances, angles and torsion?*</font>

### **7. Hydrogen Bonding**

A **[hydrogen bond]()** occurs when a hydrogen atom covalently bonded to a highly electronegative atom (like N, O, or F) interacts with another electronegative atom nearby. H-bonds are crucial for maintaining protein secondary structure (alpha-helices, beta-sheets), stabilizing the ligand in the binding pocket, and driving molecular recognition. CPPTRAJ uses geometric criteria (distance and angle) to find them.

It is customary to access the changes in the hydrogen bonding network of the whole protein. However, in protein-ligand analysis, it is much cheaper and easier to assess only the Hydrogen bonds between the residues of the proteins and the ligand.

The *avgout* flags allows cpptraj to compute the occupancy of each specific hydrogen bond.

In [ ]:
%%bash
cat > hbond.in << EOF
parm stripped.complex.top
trajin traj_strip.nc
hbond :1-163 :JZ4 out hbond_lig.dat avgout hbond_avg.dat
#!hbond :78,84,87,88 :JZ4 out hbond_lig.dat avgout hbond_avg.dat #listing specific residues to assess the hbond
run
EOF

#Then Run it in your terminal:
cpptraj -i hbond.in

- **Hbond Plots**

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

data = np.loadtxt('hbond_lig.dat', comments='#')
time_ns = data[:, 0] / 1000
hbonds = data[:, 1]
plt.figure(figsize=(8, 5))
plt.plot(time_ns, hbonds, color='navy', linewidth=1.0, alpha=0.8)
plt.title('Total Hydrogen Bonds Over Time', fontsize=14)
plt.xlabel('Time (ns)', fontsize=12, )
plt.ylabel('Number of H-Bonds', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('hbond_plot.png', dpi=300)
plt.show()

<font color="darkred">*How did the Hbond network of the ligand change over time?*</font>

<font color="darkred">*Hbond is directional, how will the plot look like if we switch the order in the hbond.in, i.e 'hbond :JZ4 :1-163' instead of 'hbond :1-163 :JZ4'? Try it and plot it, lets see*</font>

### **8. Secondary structure Analysis**

The **[Secondary Structure Analysis]()** calculates how the 3D structure of your protein evolves over time, specifically looking at alpha-helices (α), beta-sheets (β), turns, and random coils. It helps you answer: Did the protein unfold during the simulation? Did a helix melt into a random coil? Did a beta-sheet remain stable?

In [ ]:
%%bash
cat > secstruct.in << EOF
parm stripped.complex.top
trajin traj_strip.nc
secstruct sumout secstruct.dat
run
EOF

#Then Run it in your terminal:
cpptraj -i secstruct.in

- **Plots**

Secondary structure is best plotted as a "heatmap" or "timeline" plot

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Load data
data = np.loadtxt("secstruct.dat", skiprows=2)

residues = data[:, 0].astype(int)

ss = data[:, 1:]

ss_labels = ["Extended", "Bridge", "3-10", "Alpha", "Pi", "Turn", "Bend"]

plt.figure(figsize=(15, 4))
im = plt.imshow(ss.T, aspect="auto", origin="lower", cmap="viridis", vmin=0, vmax=1)
plt.yticks(range(len(ss_labels)), ss_labels)
step = max(1, len(residues)//20)
plt.xticks(range(0, len(residues), step), residues[::step])

plt.xlabel("Residue Number")
plt.ylabel("Secondary Structure")
plt.title("Secondary Structure")

cbar = plt.colorbar(im)
cbar.set_label("Occupancy")

plt.tight_layout()
plt.savefig("secondary_structure_heatmap.png", dpi=300)
plt.show()

### **9. Snapshots**

When we want to look at specific changes in our proteins at a given time or time ranges, we extract snapshots. Snapshots are specific, individual frames extracted from your trajectory. Instead of looking at the whole movie, you pick specific moments in time (e.g., at 0 ns, 50 ns, and 100 ns) to visually inspect how the protein or ligand has moved, folded, or interacted.

We will visualize the first, middle and endframes of our trajectory. The output is in pdb formats by specifying the *'pdb'* flag in the input file.

In [ ]:
%%bash
cat > snapshots.in << EOF
parm stripped.complex.top
trajin traj_strip.nc
trajout frame_0ns.pdb pdb onlyframes 1
trajout frame_5ns.pdb pdb onlyframes 5000
trajout frame_10ns.pdb pdb onlyframes 10000
run
EOF

#Then Run it in your terminal:
cpptraj -i snapshots.in



**We will align the snapshots**

In [ ]:
%%bash
cat > aligned_snapshots.in << EOF
parm stripped.complex.top
reference frame_0ns.pdb
trajin frame_0ns.pdb
trajin frame_5ns.pdb
trajin frame_10ns.pdb
rms reference @CA
trajout aligned_snapshots.pdb pdb 
run
EOF

In [ ]:
!cpptraj -i aligned_snapshots.in

*We will visulize the frames using MDTRAJ, another powerful tool for trajectory analysis after MD simulations*

In [ ]:
import nglview as nv

view = nv.show_mdtraj(traj)
view

<font color="darkred">*Did you oberve any changes in conformation of the protein across these three snapshots?*</font>

### **Summary and Reflections**

This workflow processes MD trajectories to assess protein-ligand stability. It covers PBC corrections, system alignment, global stability (RMSD, Rg, SASA), local flexibility (RMSF), secondary structure (DSSP), and specific interactions (H-bonds, contacts, ligand RMSD). Together, these analyses confirm structural integrity, identify flexible regions, and map key binding interactions.

### **Citations**

- https://ambermd.org/doc12/Amber25.pdf
- https://ambermd.org/tutorials/
- https://doi.org/10.1002/jcc.70350
- https://github.com/SHEDOOMTC/AmberTools-Analysis-Quick.git
